# 3 · Reproducing the papers

Compact, interactive versions of the three reproduction recipes in
[`reproductions/`](../reproductions). Each runs offline on synthetic data of the
same shape as the paper's dataset. To reproduce the *published numbers*, swap the
`make_*` generator for `nextaire_tools.load_table(...)` of the real data.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "reproductions"))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from _synthetic import make_graz_hourly, make_zagreb_daily, PAHS, METALS, ZAGREB_STATIONS

## Paper 1 — Petrić et al. (2024), AAQR

Winsorise → iterative impute → wind decomposition → 12 h median lags →
correlation filter → Random Forest, scored with R² and the IQR-normalised
nMAE / nRMSE on a one-year temporal hold-out.

In [2]:
from nextaire_tools import (Pipeline, MissingValueHandler, OutlierHandler, TemporalFeatures,
                   WindDecomposer, LagFeatures, CorrelationFilter, Scaler)
from nextaire_tools.models import make_regressor, regression_metrics, temporal_train_test_split

TARGET = "no2"
raw = make_graz_hourly(n_days=365, seed=0)

pipe = Pipeline([
    OutlierHandler(columns=["no", "no2", "o3", "pm10"], method="rolling_sigma", window=72, sigma=4.0),
    MissingValueHandler(strategy="iterative", random_state=0),
    WindDecomposer(direction_col="wind_dir", drop_original=True),
    TemporalFeatures(cyclical=("hour", "dayofweek", "dayofyear")),
    LagFeatures(columns=["no", "o3", "pm10", "temp", "wind_speed", "blh"], windows=[12], agg="median"),
    MissingValueHandler(strategy="bfill"),
    CorrelationFilter(threshold=0.9, protect=[TARGET]),
    Scaler(method="standard"),
])
clean = pipe.fit_transform(raw)

train, test = temporal_train_test_split(clean, test_size=0.25)
rf = make_regressor("random_forest", n_estimators=300)
rf.fit(train.drop(columns=[TARGET]), train[TARGET])
pred = rf.predict(test.drop(columns=[TARGET]))
regression_metrics(test[TARGET], pred, metrics=["r2", "nmae", "nrmse", "index_of_agreement", "fac2"])

{'r2': 0.8541491034858052,
 'nmae': 0.22756350562622638,
 'nrmse': 0.28623898531776504,
 'index_of_agreement': 0.9598510359347698,
 'fac2': 0.6424657534246575}

## Paper 2 — Jiménez-Navarro et al. (2024), Results in Engineering

Multi-target hourly forecasting scored with **blocked** cross-validation and the
WAPE metric, across a small model zoo.

In [3]:
from sklearn.linear_model import BayesianRidge
from nextaire_tools.models import BlockingTimeSeriesSplit

pipe2 = Pipeline([
    MissingValueHandler(strategy="iterative", estimator=BayesianRidge(), random_state=0),
    WindDecomposer(direction_col="wind_dir", drop_original=True),
    TemporalFeatures(cyclical=("hour", "dayofweek")),
    LagFeatures(columns=["no", "temp", "wind_speed"], windows=[12], agg="median"),
    MissingValueHandler(strategy="bfill"),
    Scaler(method="standard"),
])
clean2 = pipe2.fit_transform(make_graz_hourly(n_days=300, seed=1))

TARGETS = ["no2", "o3", "pm10"]
cv = BlockingTimeSeriesSplit(n_splits=4)
rows = {}
for name, kw in {"decision_tree": {"max_depth": 8}, "random_forest": {"n_estimators": 150}}.items():
    wape = []
    for target in TARGETS:
        X = clean2.drop(columns=TARGETS)
        for tr, te in cv.split(X):
            m = make_regressor(name, **kw).fit(X.iloc[tr], clean2[target].iloc[tr])
            wape.append(regression_metrics(clean2[target].iloc[te], m.predict(X.iloc[te]), metrics=["wape"])["wape"])
    rows[name] = float(np.mean(wape))
print("mean WAPE across targets & folds:")
rows

mean WAPE across targets & folds:


{'decision_tree': 1.017038545516695, 'random_forest': 0.8923856876238002}

## Paper 3 — Račić et al. (2026), Atmospheric Environment: X

NMF source apportionment (rank 2) plus a Random Forest on log-concentration,
explained with TreeSHAP.

In [4]:
from nextaire_tools.models import NMFApportionment

zag = make_zagreb_daily(seed=0)
station = ZAGREB_STATIONS[0]
loadings = NMFApportionment(n_components=2, random_state=0, max_iter=1000).fit(
    zag.loc[zag["station"] == station, list(PAHS)]
).loadings()
print(f"NMF rank-2 PAH loadings at {station} (factor by species):")
loadings.round(3)

NMF rank-2 PAH loadings at IMI (factor by species):


/Users/mariolovric/miniconda3/envs/cld/lib/python3.11/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


,BaP,BaA,BbF,BkF,Chry,Flu,Pyr
factor_1,3.743,1.955,1.407,1.392,1.240,1.359,1.195
factor_2,0.000,0.918,1.217,1.215,1.307,1.206,1.272


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

feats = ["temp", "radiation", "rh", "wind_speed", "pm10", "no2", "gas_sum", "traffic", "julian", "dow", "month"]
dummies = pd.get_dummies(zag["station"], prefix="station").astype(float)
X = pd.concat([zag[feats], dummies], axis=1)
y = np.log(zag["BaP"].to_numpy())

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
rf = make_regressor("random_forest", n_estimators=200).fit(X_tr, y_tr)
print("BaP  R2 =", round(regression_metrics(y_te, rf.predict(X_te), metrics=["r2"])["r2"], 3))

BaP  R2 = 0.491


In [6]:
# TreeSHAP feature ranking (needs the optional 'shap' extra: pip install 'nextaire_tools[shap]').
try:
    from nextaire_tools.models import shap_importance
    imp = shap_importance(rf, X_te)
    display(imp.head(6))
except ImportError:
    print("shap not installed — skipping TreeSHAP")

,feature,mean_abs_shap
0,gas_sum,0.506655
1,traffic,0.070016
2,temp,0.066540
3,rh,0.018229
4,pm10,0.015844
5,no2,0.014988
